In [ ]:
import numpy as np 
import pandas as pd 

In [2]:
from xgboost import XGBClassifier

In [ ]:
file_paths = [
    r"../Data_money_luandring/HI-Large_Trans.csv",
    r"../Data_money_luandring/LI-Large_Trans.csv"
]

filtered_data = []

for file_path in file_paths:
    chunks = pd.read_csv(file_path, chunksize=100000)

    for chunk in chunks:
        filtered_chunk = chunk[chunk["Is Laundering"] == 1]
        filtered_data.append(filtered_chunk)

df_1 = pd.concat(filtered_data, ignore_index=True)

print("Fraud shape:", df_1.shape)

Fraud shape: (326150, 11)


In [ ]:
n_samples = df_1.shape[0]

sampled_zeros = []

for file_path in file_paths:
    chunks = pd.read_csv(file_path, chunksize=100000)

    for chunk in chunks:
        zeros = chunk[chunk["Is Laundering"] == 0]

        if not zeros.empty:
            sampled_zeros.append(zeros.sample(frac=0.01))  # سحب عشوائي بسيط

df_0_all = pd.concat(sampled_zeros)


df_0 = df_0_all.sample(n=n_samples, random_state=42)

print("Non-Fraud shape:", df_0.shape)

Non-Fraud shape: (326150, 11)


In [ ]:
df_balanced = pd.concat([df_0, df_1])
df_balanced = df_balanced.sample(frac=1, random_state=42) 

In [118]:
df_balanced["Is Laundering"].value_counts()

Is Laundering
1    326150
0    326150
Name: count, dtype: int64

In [119]:
df_balanced.columns = df_balanced.columns.str.strip()

In [120]:
df_balanced.info()

<class 'pandas.core.frame.DataFrame'>
Index: 652300 entries, 263792 to 11205151
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Timestamp           652300 non-null  object 
 1   From Bank           652300 non-null  int64  
 2   Account             652300 non-null  object 
 3   To Bank             652300 non-null  int64  
 4   Account.1           652300 non-null  object 
 5   Amount Received     652300 non-null  float64
 6   Receiving Currency  652300 non-null  object 
 7   Amount Paid         652300 non-null  float64
 8   Payment Currency    652300 non-null  object 
 9   Payment Format      652300 non-null  object 
 10  Is Laundering       652300 non-null  int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 59.7+ MB


In [121]:
target_col = "Is Laundering"

df_balanced["Timestamp"] = pd.to_datetime(df_balanced["Timestamp"], errors="coerce")

df_balanced["year"] = df_balanced["Timestamp"].dt.year
df_balanced["month"] = df_balanced["Timestamp"].dt.month
df_balanced["day"] = df_balanced["Timestamp"].dt.day
df_balanced["hour"] = df_balanced["Timestamp"].dt.hour

In [122]:
def group_rare_categories(df_balanced, col, top_n=10):
    top_categories = df_balanced[col].value_counts().nlargest(top_n).index
    df_balanced[col] = df_balanced[col].where(df_balanced[col].isin(top_categories), "Others")
    return df_balanced

for col in ["Receiving Currency", "Payment Currency", "Payment Format"]:
    if col in df_balanced.columns :
        df_balanced = group_rare_categories(df_balanced, col, top_n=10)

In [123]:
freq_cols = [c for c in ["From Bank", "To Bank", "Account", "Account.1"] if c in df_balanced.columns]

for col in freq_cols:
    freq_map = df_balanced[col].value_counts(dropna=False).to_dict()
    df_balanced[col + "_freq"] = df_balanced[col].map(freq_map)

df_balanced = df_balanced.drop(columns=freq_cols)


In [124]:
df_balanced.head(20)

,Timestamp,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,year,month,day,hour,From Bank_freq,To Bank_freq,Account_freq,Account.1_freq
263792,2022-09-09 14:17:00,940.10,US Dollar,940.10,US Dollar,ACH,1,2022,9,9,14,77,182,3,3
5400234,2022-08-02 11:02:00,722.04,US Dollar,722.04,US Dollar,Credit Card,0,2022,8,2,11,78888,25,14579,2
107123412,2022-09-29 12:16:00,21835.39,US Dollar,21835.39,US Dollar,Cheque,0,2022,9,29,12,1344,112,2,1
154799756,2022-10-24 13:24:00,975.07,US Dollar,975.07,US Dollar,Credit Card,0,2022,10,24,13,137,39,2,1
126694461,2022-10-09 23:59:00,72.76,Euro,72.76,Euro,Cheque,0,2022,10,9,23,1397,82,2,1
111095210,2022-09-30 00:11:00,401.79,US Dollar,401.79,US Dollar,Reinvestment,0,2022,9,30,0,10,21,1,1
85296383,2022-09-16 09:00:00,7127.90,Euro,7127.90,Euro,Credit Card,0,2022,9,16,9,63,56,1,2
78301,2022-09-24 09:30:00,205.58,Euro,205.58,Euro,ACH,1,2022,9,24,9,3,188,1,12
166468958,2022-10-31 16:44:00,56228775.72,Ruble,56228775.72,Ruble,Cash,0,2022,10,31,16,174,94,1,1
20215252,2022-08-10 17:16:00,18226.39,Yuan,18226.39,Yuan,Credit Card,0,2022,8,10,17,197,48,1,1


In [ ]:
from sklearn.model_selection import train_test_split

X = df_balanced.drop(["Is Laundering","Timestamp"], axis=1)
y = df_balanced["Is Laundering"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,        
    random_state=42,       
    shuffle=True,        
    stratify=y             
)

In [126]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [127]:
categorical_cols = [c for c in ["Receiving Currency", "Payment Currency", "Payment Format"] if c in X_train.columns]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

In [181]:
from sklearn.pipeline import Pipeline

pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Amount Received',
                                                   'Amount Paid', 'year',
                                                   'month', 'day', 'hour',
                                                   'From Bank_freq',
                                                   'To Bank_freq',
                                                   'Account_freq',
                                                   'Account.1_freq']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_freq...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [ ]:
from sklearn.metrics import precision_recall_curve

y_prob = pipe.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

beta = 2
f2 = (1 + beta**2) * (precision * recall) / ((beta**2 * precision) + recall + 1e-8)

best_idx = np.argmax(f2)
best_threshold = thresholds[best_idx]

print("Best Threshold:", best_threshold)

Best Threshold: 0.25511903


In [188]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

y_pred = (y_prob > best_threshold).astype(int)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.99      0.81      0.89     81538
           1       0.84      0.99      0.91     81537

    accuracy                           0.90    163075
   macro avg       0.92      0.90      0.90    163075
weighted avg       0.92      0.90      0.90    163075

Confusion Matrix:
 [[66169 15369]
 [  665 80872]]
ROC-AUC: 0.9694948922219842
PR-AUC: 0.966773557309109


In [189]:
import joblib
import json
import os

deploy_dir = r"C:\Users\Admin\Desktop\AI_detect_money_API\deployment_package"
os.makedirs(deploy_dir, exist_ok=True)


joblib.dump(pipe, os.path.join(deploy_dir, "xgb_model.joblib"))


with open(os.path.join(deploy_dir, "threshold.json"), "w") as f:
    json.dump({"threshold": float(best_threshold)}, f)


feature_columns = X_train.columns.tolist()
with open(os.path.join(deploy_dir, "feature_columns.json"), "w") as f:
    json.dump(feature_columns, f)

print("Model saved successfully")

Model saved successfully
